In [1]:
import tensorflow as tf
import TensorSlider as ts
import keras


2025-02-06 18:40:26.873147: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1738863626.883519   78913 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1738863626.886644   78913 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-02-06 18:40:26.898359: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
print(tf.config.list_physical_devices('GPU'))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [3]:
def createLabelsBatch(data, lookforward):
    """
    create input labels from the lookahead data
    """

    shape = lookforward.shape

    dividepricesby = tf.reshape(lookforward[:,0,0,0], (shape[0], 1, 1, 1))
    prices = tf.divide(lookforward[:,:,0:4], dividepricesby) # divide by latest base timeframe close
    prices -= 1 # zero out
    prices *= 10 # convert to 1/10 percentage, so 1 = 10 percent

    low = tf.reduce_min(prices[:,0,2], 1)
    high = tf.reduce_max(prices[:,0,1], 1)

    label = tf.stack([low, high], axis = 1)



    shape = data.shape
    # data processing, format: batch, timeframe, feature(ohlcv), window (ascending time)
    # prices
    dividepricesby = tf.reshape(data[:,0,3,-1], (shape[0], 1, 1, 1))
    prices = tf.divide(data[:,:,0:4], dividepricesby) # divide by latest base timeframe close
    prices -= 1 # zero out
    prices *= 10 # convert to 1/10 percentage, so 1 = 10 percent
    
    volumes = tf.reshape(data[:,:,4], (shape[0], shape[1], 1, shape[3]))
    dividebyvolume = tf.reshape(data[:,:,4,-1], (shape[0], shape[1], 1, 1))
    volume = tf.divide(volumes, dividebyvolume) # divide by latest volume (of each timeframe)
    volume -= 1
    volume *= 10 # convert to 1/10 percentage, so 1 = 10 percent
    
    data = tf.clip_by_value(tf.concat([prices, volume], axis=2), -2, 2)
    # remove nans
    data, label = tf.keras.ops.nan_to_num(data), tf.keras.ops.nan_to_num(label)

    return data, label

def decode(record_bytes):
    # Function for parsing each record in the tf files
    example = tf.io.parse_single_example(
        # Data
        record_bytes,

        # Schema
        {
        'Timeframe': tf.io.FixedLenFeature([], tf.string),
        'timestamp': tf.io.RaggedFeature(dtype=tf.int64),
        'Open': tf.io.RaggedFeature(dtype=tf.float32),
        'High': tf.io.RaggedFeature(dtype=tf.float32),
        'Low': tf.io.RaggedFeature(dtype=tf.float32),
        'Close': tf.io.RaggedFeature(dtype=tf.float32),
        'Volume': tf.io.RaggedFeature(dtype=tf.float32),
        }
        )

    return example

def getDataset(path):
    ds = tf.data.TFRecordDataset(path,  num_parallel_reads = tf.data.AUTOTUNE)
    ds = ds.map(decode, num_parallel_calls = tf.data.AUTOTUNE)
    return ds

## Get Datasets

In [4]:
tfrecordpath = "../Data/tfrecords/"

windowsize = 100
lookahead = 5
batch_size = 100

def getSlider(coin):

    path = tfrecordpath + coin+"/{tframe}.tfrecord"
    datasets = {"1m" : getDataset(path.format(tframe="1m")),
                "5m":  getDataset(path.format(tframe="5m")),
                "15m":  getDataset(path.format(tframe="15m")),
                "30m":  getDataset(path.format(tframe="30m")),
                "1h":  getDataset(path.format(tframe="1h"))}
    return ts.WindowSlider(datasets, windowsize, lookahead, batch_size)

def getSliders(coins):
    datasets = []
    for coin in coins:
        datasets.append(tf.data.Dataset.from_generator(lambda: getSlider(coin),
            output_signature=(
                tf.TensorSpec((batch_size,5,5,windowsize), dtype=tf.float32),
                tf.TensorSpec((batch_size,5,5, lookahead+1), dtype=tf.float32))
            ))#.prefetch(tf.data.AUTOTUNE))
    return datasets

coins = ["BTCUSD_PERP", "ETHUSD_PERP", "ADAUSD_PERP", "SOLUSD_PERP"]
datasets = getSliders(coins)

I0000 00:00:1738863628.625237   78913 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5592 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3060 Ti, pci bus id: 0000:05:00.0, compute capability: 8.6


## Combine Datasets

In [8]:
ds = tf.data.Dataset.sample_from_datasets(datasets = datasets).prefetch(tf.data.AUTOTUNE)

ds = ds.map(createLabelsBatch, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

In [9]:
import keras
import os

def load_model(name):
    """
    Load model and latest checkpoint (if applicable). Returns model, and last epoch that was trained.
    If no checkpoints present, either creates the checkpoint folder or trains directly on the saved model.
    """
    folder = "models/" + name + "/"
    # Load model
    model = keras.models.load_model(folder + "model.keras")

    # Check if we have any checkpoints in the first place
    if not os.path.exists(folder + "checkpoints"):
        # No checkpoint folder, lets create one, and return the model, epoch 0
        os.mkdir(folder + "checkpoints")
        return model, 0

    # Check how many epoch checkpoints we have in the (existing!) checkpoint folder
    checkpoints = os.listdir("models/" + name + "/checkpoints/")

    # We do this by just counting how many files are in there, we assume there will be no vandalism
    # and all files inside the checkpoint folder are checkpoints
    lastEpoch = len(checkpoints)

    # load the latest checkpoint if we have more than 1 of them
    if lastEpoch >= 1:
        model.load_weights("models/" + name + "/checkpoints/" + str(lastEpoch) + ".weights.h5")

    return model, lastEpoch

class saveEachEpoch(tf.keras.callbacks.Callback):
    """
    custom callback for saving checkpoints because of course we need to do this on our own
    """

    def __init__(self, name, lastEpoch=0):
        super().__init__()
        # no idea if we want to or need to super this
        self.lastEpoch = lastEpoch
        self.name = name
        print("Model was trained for " + str(lastEpoch) + " epochs before.")

    def on_epoch_end(self, epoch, logs):
        """
        Create a checkpoint and save.
        """
        # Increment which epoch this is
        self.lastEpoch += 1
        # Save weights
        self.model.save_weights("models/" + str(self.name) + "/checkpoints/" + str(self.lastEpoch) + ".weights.h5")
        print("\nSaved checkpoint of epoch number " + str(self.lastEpoch))
        # Save the latest model
        self.model.save("models/" + str(self.name) + "/model.keras")


In [10]:
modelName = "modeltest"

tensorboard = keras.callbacks.TensorBoard(
                                                log_dir=f"models/{modelName}/logs",
                                                histogram_freq=100,
                                                write_graph=True,
                                                write_images=False,
                                                write_steps_per_second=True,
                                                update_freq="batch",
                                                #profile_batch = '70,100',
                                                embeddings_freq=0,
                                                embeddings_metadata=None,
                                            )

model, epochs = load_model(modelName)

history = model.fit(ds, epochs=10, verbose=1, validation_data=None, callbacks=[saveEachEpoch(modelName, epochs), tensorboard])

Model was trained for 3 epochs before.
Epoch 1/10
  13120/Unknown 107s 8ms/step - MeanAbsolutePercentageError: 1940768.3750 - MeanSquaredError: 0.0019 - loss: 0.0019Index maxxed, index: 0, updateTo: 1791137
  13182/Unknown 107s 8ms/step - MeanAbsolutePercentageError: 1940070.6250 - MeanSquaredError: 0.0019 - loss: 0.0019Index maxxed, index: 0, updateTo: 1791137
  13328/Unknown 108s 8ms/step - MeanAbsolutePercentageError: 1938423.8750 - MeanSquaredError: 0.0019 - loss: 0.0019Index maxxed, index: 0, updateTo: 1791137
  13362/Unknown 109s 8ms/step - MeanAbsolutePercentageError: 1938043.0000 - MeanSquaredError: 0.0019 - loss: 0.0019Index maxxed, index: 0, updateTo: 1791137
  13382/Unknown 109s 8ms/step - MeanAbsolutePercentageError: 1937820.2500 - MeanSquaredError: 0.0019 - loss: 0.0019

2025-02-06 18:44:47.552376: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
	 [[IteratorGetNext/_2]]



Saved checkpoint of epoch number 4
13384/13384 ━━━━━━━━━━━━━━━━━━━━ 110s 8ms/step - MeanAbsolutePercentageError: 1937787.0000 - MeanSquaredError: 0.0019 - loss: 0.0019
Epoch 2/10
13119/13384 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - MeanAbsolutePercentageError: 1941231.3750 - MeanSquaredError: 0.0024 - loss: 0.0024Index maxxed, index: 0, updateTo: 1791137
13182/13384 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - MeanAbsolutePercentageError: 1940485.8750 - MeanSquaredError: 0.0024 - loss: 0.0024Index maxxed, index: 0, updateTo: 1791137
13328/13384 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - MeanAbsolutePercentageError: 1938755.8750 - MeanSquaredError: 0.0024 - loss: 0.0024Index maxxed, index: 0, updateTo: 1791137
13363/13384 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - MeanAbsolutePercentageError: 1938344.0000 - MeanSquaredError: 0.0024 - loss: 0.0024Index maxxed, index: 0, updateTo: 1791137
13384/13384 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - MeanAbsolutePercentageError: 1938098.5000 - MeanSquaredError: 0.0024 - loss: 0.0024
S

KeyboardInterrupt: 